<img src="./images/hsph.png" alt="HSPH Logo" width="400"><br>

# Lab 2 - RAG and Prompt Engineering to Extract Real-World Phenotypes (EPI 264)

This notebook introduces **prompt basics** using a local LLM (via Ollama + LangChain), and then applies prompts to a subset of **real (de-identified) clinical notes** prepared in Lab 6.

## Learning goals
By the end of this notebook, you will be able to:
- Send **system + user** messages to an LLM
- Use **prompt templates** (parameterized prompts) for repeatable workflows
- Apply a structured extraction prompt to real notes from our Lab 6 cohort

> Notes: We are not building embeddings or retrieval yet. This notebook is about **prompting** and **structured extraction**.

In [ ]:
# -----------------------------------------------------------
# 1. Load the Ollama Model
# -----------------------------------------------------------
# This cell loads a local Large Language Model (LLM) using LangChain's
# `ChatOllama` wrapper. Make sure the model has been pulled via Ollama CLI before use.

# To explore available models, visit:
# - Ollama Library: https://ollama.com/library
# - Hugging Face (for embeddings and additional models): https://huggingface.co/models

from langchain_ollama import ChatOllama

# Define the model name — make sure this model is already downloaded using:
#   ollama pull deepseek-v2:16b
model_name = "qwen2"  # Alternatives: "qwen2", "llama3", "gpt-oss:20b" etc.

# Initialize the model with LangChain
model = ChatOllama(model=model_name)

print(f"✅ Model '{model_name}' is loaded and ready to use.")


## 2. Basic Prompt Interaction (System + User Messages)

In this section, you will run a simple clinical query to confirm your local model is working.

**Key idea:**
- The **system message** sets role + behavior (“how to answer”).
- The **user message** asks the question (“what to answer”).

<img src="./images/basic_prompt.png" alt="i2b2 Logo" width="900">


In [ ]:
# -----------------------------------------------------------
# 2. Run a Simple Clinical Query with System + User Prompts
# -----------------------------------------------------------
# This tests the model by asking a simple clinical question using structured messages.

from langchain_core.messages import HumanMessage, SystemMessage
from IPython.display import display, Markdown



messages = [
    SystemMessage(content=(
        "You are a knowledgeable medical provider. "
        "Provide clear, evidence-based explanations about a medical conditions."
    )),
    HumanMessage(content="What is dementia? What are its common symptoms and treatments?")
]

# Run inference
response = model.invoke(messages)

# Display result
display(Markdown("### Model Response:"))
display(Markdown(response.content))


## 3. Prompt Templates (Reusable Prompts)

Hard-coding prompts works for one-off queries, but research workflows need **repeatable prompts**.

In this section, you will use `ChatPromptTemplate` to:
- Define a prompt once (with placeholders)
- Reuse it across conditions / patient contexts
- Standardize outputs across runs (important for reproducibility)

<img src="./images/prompt_template.png" alt="Prompt Template" width="900">


In [ ]:
# -----------------------------------------------------------
# 3.1. Create a Reusable Prompt Template (Dynamic Querying)
# -----------------------------------------------------------
# This cell demonstrates how to build a dynamic prompt template using placeholders
# for different medical conditions and patient profiles. The prompt is populated
# with variables at runtime and sent to the LLM for inference.

from langchain_core.prompts import ChatPromptTemplate

# Define input variables
patient_type = "5-year old child"
disease = "dementia"

# Define a role-based prompt using variable placeholders
messages = [
    ("system",
     "You are a knowledgeable medical provider. Provide clear, evidence-based explanations appropriate for a {patient_type} patient."),

    ("human",
     "What is {disease}? What are its common symptoms and treatments?")
]

# Create a prompt template with dynamic inputs
prompt_template = ChatPromptTemplate.from_messages(messages)

# Fill the template with actual values
prompt = prompt_template.invoke({
    "patient_type": patient_type,
    "disease": disease
})

# Run the model with the constructed prompt
response = model.invoke(prompt)

# Display the generated answer
display(Markdown("### AI Response"))
display(Markdown(response.content))


In [ ]:
# -----------------------------------------------------------
# 3.2. Prompt Template for Dementia Phenotype Extraction (Yes/No)
# -----------------------------------------------------------
# Goal:
# Demonstrate that an LLM can extract a phenotype from unstructured text.
# -----------------------------------------------------------

from langchain_core.prompts import ChatPromptTemplate

messages_notes = [
    ("system",
     "You are an advanced clinical phenotyping assistant. "
     "Your task is to extract key clinical details from unstructured notes in a clear, structured format.\n"
     "Rules:\n"
     "- Use ONLY information explicitly present in the note. Do NOT infer or invent.\n"
     "- If the note is administrative/non-clinical (e.g., 'created in error', 'dictated', 'reconcile meds'), still output sections 1–4, but keep them minimal.\n"
     "- Do NOT include meta statements about redaction, privacy, dates shifted, or removed identifiers.\n"),

    ("human",
     "Analyze the following clinical note:\n\n"
     "{patient_note}\n\n"
     "Extract the following sections using numbered headings and bullet points:\n"
     "1. Patient demographics (only what is stated)\n"
     "2. Chief complaints / reason for visit\n"
     "3. Current medications (only those explicitly listed)\n"
     "4. Dementia phenotype (Yes/No)\n"
     "   - Answer 'Yes' ONLY if dementia is documented (e.g., 'dementia', 'Alzheimer', "
     "'vascular dementia', 'Lewy body dementia') OR clearly stated as a prior diagnosis.\n"
     "   - If dementia is not explicitly mentioned, answer 'No'. Do NOT infer from memory complaints alone.\n"
     "   - Under section 4, include these sub-bullets (one per line):\n"
     "     • Phenotype: Yes/No\n"
     "     • Rationale: 1–2 sentences grounded ONLY in the evidence\n"
     "     • Confidence: low/medium/high\n\n"
     "Important: Keep the output cleanly separated by sections 1–4 "
     "(do not combine evidence/rationale/confidence into one line).")
]

prompt_template_notes = ChatPromptTemplate.from_messages(messages_notes)

print(">> Prompt created and ready to use.")

## 4. Apply Prompts to Real Notes from the Lab 6 Cohort

Now we will load:
- the cleaned notes dataset (deduplicated + restricted to the chosen 2-year window in Lab 6)
- the structured dementia flags + physician gold label

We will:
1. Merge notes with patient-level labels
2. Inspect notes for one patient
3. Run a structured extraction prompt on a selected note

> Reminder: These notes are de-identified synthetic narratives derived from original notes, designed for teaching and downstream RAG.

In [ ]:
# -----------------------------------------------------------
# 4.1. Load Cleaned Notes and Structured Evaluation Dataset
# -----------------------------------------------------------
# From Lab 6:
# - lab6_clean_notes_baseline.parquet
# - lab6_structured_dementia_eval.csv
# -----------------------------------------------------------

# Purpose:
# Load the data into a pandas DataFrame and inspect the structure before decoding.

from pathlib import Path
import pandas as pd

filepath = Path("data_files")

notes = pd.read_parquet(filepath / "lab6_clean_notes_baseline.parquet")
eval_df = pd.read_csv(filepath / "lab6_structured_dementia_eval.csv")

print("Notes shape:", notes.shape)
print("Eval dataset shape:", eval_df.shape)

display(notes.head(10))
display(eval_df.head(10))


In [ ]:
# -----------------------------------------------------------
# 4.2. Merge Notes with Structured & Gold Labels
# -----------------------------------------------------------

notes_eval = notes.merge(
    eval_df,
    on="patient_num",
    how="inner"
)

print("Merged dataset shape:", notes_eval.shape)
display(notes_eval.head(10))

In [ ]:
# -----------------------------------------------------------
# 4.3. Inspect First 5 Notes for a Specific Patient
# -----------------------------------------------------------

from IPython.display import display, Markdown

sample_patient = "H122074591"

sample_notes = notes_eval[
    notes_eval["patient_num"] == sample_patient
].head(5)

print(f"Showing first {sample_notes.shape[0]} notes for patient {sample_patient}")

# Display metadata table
display(sample_notes[[
    "patient_num",
    "visit_id",
    "note_csn_id",
    "inpatient_note_type_dsc",
    "create_dts_shifted",
    "baseline_dementia",
    "gold_dementia"
]])

# Display text for first 10 notes
for _, row in sample_notes.iterrows():
    display(Markdown(f"""
---
### Note CSN ID: {row['note_csn_id']}
**Visit ID:** {row['visit_id']}
**Note Type:** {row['inpatient_note_type_dsc']}
**Date:** {row['create_dts_shifted']}
**Structured Dementia:** {row['baseline_dementia']}
**Gold Dementia:** {row['gold_dementia']}

---

{row['note_txt_deid']}
"""))

In [ ]:
# -----------------------------------------------------------
# 4.4. Analyze a Real Patient Note Using note_csn_id
# -----------------------------------------------------------

from IPython.display import display, Markdown

NOTE_CSN_ID = 2650311453  # <-- set this to any note_csn_id you want

# Pull the note row
note_row = notes_eval.loc[notes_eval["note_csn_id"] == NOTE_CSN_ID].iloc[0]

# Extract the note text
patient_note_text = note_row["note_txt_deid"]

# Fill prompt
filled_prompt = prompt_template_notes.invoke({"patient_note": patient_note_text})

# Run model
clinical_response = model.invoke(filled_prompt)

# Display context + output
display(Markdown(f"""
### Note Selected
- **Patient:** {note_row['patient_num']}
- **Note CSN ID:** {note_row['note_csn_id']}
- **Note Type:** {note_row['inpatient_note_type_dsc']}
"""))

display(Markdown("### Extracted Clinical Information"))
display(Markdown(clinical_response.content))